# 滑动窗口记忆

> **只保留最近 *k* 个轮次：在保持对话连贯的同时，这是控制记忆成本最直接的方法。**

在[上一个 notebook](../01_conversation_buffer_memory/conversation_buffer_memory.ipynb)里，我们看到**对话缓冲记忆**会保存所有内容。这让每轮的输入 token 线性增长，累计成本平方级增长。对大多数聊天应用来说，这很浪费：用户很少会提 30 轮之前说过的话。

想象一块只贴得下五张便利贴的白板。贴第六张时，你会撕掉最旧的那张扔掉。**滑动窗口记忆**就是这样工作的：它保留一个固定大小的最近消息窗口，丢掉所有更旧的消息。最近 *k* 条消息始终可用，更旧的消息则一去不回。

**读完本 notebook，你会理解：**
- 如何用 Anthropic SDK 从零构建一个基于 `deque` 的滑动窗口。
- 窗口大小、回忆能力和 token 成本之间精确的权衡。
- 如何运行一个小型评估，测量不同窗口大小下的回忆表现。
- 滑动窗口记忆什么时候是正确（或错误）的选择。

## 关键概念

- **窗口大小 *k***：保留消息的最大数量。*k* 越大，回忆效果越好，token 成本也越高。
- **FIFO 淘汰**：FIFO 即「先进先出」（first-in, first-out）。当 `len(messages) > k` 时，最旧的消息从队首掉出，窗口随时间向前「滑动」。
- **`collections.deque`**：Python 的双端队列，带 `maxlen` 参数，天然适合这个场景：向已满的 deque 添加元素时，最旧的元素会自动掉出。
- **新近度偏差**：按设计，智能体记得较新的上下文，忘掉较旧的。当最近的上下文最重要时，这是特性而不是缺陷。
- **按消息计数与按轮次计数的窗口**：大小为 *k* 的消息窗口逐条计数（用户和助手消息各算一条）。大小为 *k* 的轮次窗口完整保留最后 *k* 组「用户-助手」对话对。轮次窗口能避免孤立消息（只剩回答、没有提问）。
- **恒定的 token 成本**：与缓冲记忆不同，每轮 token 成本由窗口大小封顶，带来可预测的延迟和成本。

## 架构

<p align="center">
  <img src="../../images/diagrams/02_sliding_window_memory.svg" alt="架构图" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
sequenceDiagram
    participant U as User
    participant W as Sliding Window (k=4 messages)
    participant L as LLM (Claude)

    Note over W: Window: []

    U->>W: "Hi, I'm Alice" (msg 1)
    W->>L: [msg1]
    L-->>W: "Hello Alice!" (msg 2)
    Note over W: Window: [msg1, msg2]

    U->>W: "I like Python" (msg 3)
    W->>L: [msg1, msg2, msg3]
    L-->>W: "Python is great!" (msg 4)
    Note over W: Window: [msg1, msg2, msg3, msg4] - FULL

    U->>W: "What's my name?" (msg 5)
    Note over W: ⚠ Window full → drop msg1
    W->>W: Evict msg1, append msg5
    W->>L: [msg2, msg3, msg4, msg5]
    L-->>W: "Your name is Alice." (msg 6)
    Note over W: Drop msg2 → Window: [msg3, msg4, msg5, msg6]

    U->>W: "What language do I like?"
    Note over W: msg3 ("I like Python") still<br/>in window → agent remembers
    W->>L: [msg4, msg5, msg6, msg7]
    L-->>W: "You said you like Python!"
    Note over W: Eventually msg3 slides out<br/>and that fact is forgotten
```

</details>

In [ ]:
# 安装所需的包（只需运行一次）
%pip install -q python-dotenv matplotlib pandas deepagents langchain-openai

## 环境准备

导入 Anthropic SDK 和标准辅助工具。我们还会用到 Python `collections` 模块里的 `deque`。**deque**（读作「deck」）是一种双端队列，可以容纳固定数量的元素。再添加一个时，最旧的元素会自动掉出。

In [ ]:
import os
import copy
from collections import deque

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)



def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )



def usage_of(reply, prompt_messages: list[dict]) -> dict:
    """读取一轮对话的 token 用量；端点未返回时用 tiktoken 估算。"""
    import tiktoken

    usage = getattr(reply, "usage_metadata", None) or {}
    encoder = tiktoken.get_encoding("cl100k_base")
    prompt_text = "".join(str(m.get("content", "")) for m in prompt_messages)
    reply_text = to_text(reply.content)
    return {
        "input_tokens": int(usage.get("input_tokens") or len(encoder.encode(prompt_text))),
        "output_tokens": int(usage.get("output_tokens") or len(encoder.encode(reply_text))),
    }


print("✓ Model ready")


## 核心实现

关键洞察：Python 的 `collections.deque(maxlen=k)` 免费提供了滑动窗口行为。向一个已满的 deque `append()` 时，另一端最旧的元素会自动掉出。

In [ ]:
class SlidingWindowMemory:
    """滑动窗口记忆：只保留最近 *k* 条消息。"""

    def __init__(
        self,
        model,
        window_size: int = 10,
        system_prompt: str | None = None,
    ):
        # deepagents 智能体本身无状态，窗口逻辑全部留在本类里
        self.agent = create_deep_agent(model=model, system_prompt=system_prompt)
        self.model = model
        self.system_prompt = system_prompt

        # 滑动窗口：固定最大长度的 deque
        self.window_size = window_size
        self.messages: deque[dict] = deque(maxlen=window_size)

        # 记录所有消息用于分析（不发送给 LLM）
        self.full_history: list[dict] = []
        self.turn_token_usage: list[dict] = []


接下来添加 `chat` 方法和检查辅助方法。`chat` 方法在把消息发给 API 前，会先把 deque 转成普通列表。它还会记录每一轮的 token 用量和当时的窗口大小。检查辅助方法让你能对比 LLM 看到的内容（窗口）和完整历史。

In [ ]:
# ── 对话 ──────────────────────────────────────────────────────
def _chat(self, user_input: str) -> str:
    """发送一条消息；只有最近 *k* 条消息会发给 LLM。"""
    user_msg = {"role": "user", "content": user_input}
    self.messages.append(user_msg)
    self.full_history.append(user_msg)

    # deque 转成列表供 invoke 使用；窗口外的消息不会出现在这里
    result = self.agent.invoke({"messages": list(self.messages)})
    reply = result["messages"][-1]
    assistant_text = to_text(reply.content)

    assistant_msg = {"role": "assistant", "content": assistant_text}
    self.messages.append(assistant_msg)
    self.full_history.append(assistant_msg)

    usage = usage_of(reply, list(self.messages))
    self.turn_token_usage.append({
        "turn": len(self.turn_token_usage) + 1,
        "input_tokens": usage["input_tokens"],
        "output_tokens": usage["output_tokens"],
        "window_msgs": len(self.messages),
    })

    return assistant_text

SlidingWindowMemory.chat = _chat


窗口检查与统计辅助方法。


In [ ]:
# ── 检查辅助方法 ───────────────────────────────────────
def _get_window(self) -> list[dict]:
    """返回当前窗口内容（LLM 看到的部分）。"""
    return list(self.messages)


def _get_full_history(self) -> list[dict]:
    """返回发送过的所有消息，包括已淘汰的。"""
    return copy.deepcopy(self.full_history)


@property
def _evicted_count(self) -> int:
    """已从窗口淘汰的消息数量。"""
    return len(self.full_history) - len(self.messages)


def _clear(self) -> None:
    self.messages.clear()
    self.full_history.clear()
    self.turn_token_usage.clear()


def _repr(self) -> str:
    return (
        f"SlidingWindowMemory(model=llm, window={len(self.messages)}/{self.window_size}, "
        f"total={len(self.full_history)} messages, "
        f"evicted={self.evicted_count})"
    )


SlidingWindowMemory.get_window = _get_window
SlidingWindowMemory.get_full_history = _get_full_history
SlidingWindowMemory.evicted_count = _evicted_count
SlidingWindowMemory.clear = _clear
SlidingWindowMemory.__repr__ = _repr

print("✓ SlidingWindowMemory methods attached")


## 使用示例：观察窗口滑动

我们用一个小窗口（`k=6` 条消息，即 3 个轮次），观察对话变长时会发生什么。先在对话早期埋下一个事实，等它滑出窗口后再去问。

In [ ]:
mem = SlidingWindowMemory(
    model=llm,
    window_size=6,  # 3 个完整轮次（用户 + 助手各一条）
    system_prompt="You are a concise assistant. Reply in 1-2 sentences.",
)

conversation = [
    "My name is Alice and I'm a pilot.",           # 轮次 1：埋下一个事实
    "I fly Boeing 737s for a regional airline.",    # 轮次 2：再埋一个事实
    "What's the weather like in Seattle today?",   # 轮次 3：无关的填充内容
    "What's my name and what do I do for work?",   # 轮次 4：回忆测试（轮次 1 可能已被淘汰）
]

for msg in conversation:
    print(f"👤 User:  {msg}")
    reply = mem.chat(msg)
    print(f"🤖 Agent: {reply}")
    print(f"   📊 Window: {len(mem.messages)}/{mem.window_size} msgs | Evicted: {mem.evicted_count}")
    print()

看看 LLM 当前能看到什么、已经忘了什么。窗口内的消息对模型可见；被淘汰的消息永远消失。

In [ ]:
# 查看 LLM 当前看到的与已忘记的内容
print("=== Messages IN the window (LLM can see) ===")
for i, msg in enumerate(mem.get_window()):
    role = "USER" if msg["role"] == "user" else "ASST"
    preview = msg["content"][:70] + ("..." if len(msg["content"]) > 70 else "")
    print(f"  [{i}] {role}: {preview}")

print(f"\n=== Evicted messages (LLM cannot see): {mem.evicted_count} ===" )
evicted = mem.get_full_history()[:mem.evicted_count]
for i, msg in enumerate(evicted):
    role = "USER" if msg["role"] == "user" else "ASST"
    preview = msg["content"][:70] + ("..." if len(msg["content"]) > 70 else "")
    print(f"  [{i}] {role}: {preview}")

## 评估：不同窗口大小下的新近度与回忆

关键问题是：**窗口大小如何影响智能体回忆对话中不同位置所埋事实的能力？**

我们将运行一个受控实验：
1. 在第 1-5 轮埋下 5 个不同的事实。
2. 加入填充轮次，把较早的事实挤出较小的窗口。
3. 让模型回忆每个事实。
4. 在 k = 4、8、12、20 条消息的窗口大小下给回忆准确率打分。

In [ ]:
import re

FACTS = [
    ("My name is Elena.", "elena"),
    ("I was born in Prague.", "prague"),
    ("My favorite color is teal.", "teal"),
    ("I have two cats named Salt and Pepper.", "salt"),
    ("I work as a data scientist at a biotech company.", "data scientist"),
]

FILLER_MESSAGES = [
    "Tell me a fun fact about octopuses.",
    "What's the capital of Mongolia?",
    "How does photosynthesis work in one sentence?",
    "What year was the Eiffel Tower built?",
    "Name a famous mathematician.",
]

RECALL_QUESTIONS = [
    ("What is my name?", "elena"),
    ("Where was I born?", "prague"),
    ("What is my favorite color?", "teal"),
    ("What are my cats' names?", "salt"),
    ("What do I do for work?", "data scientist"),
]




现在定义评估函数。它先埋入事实，再用填充轮次把旧事实挤出小窗口，然后提出回忆问题。每个回答都会检查是否包含关键词，以判断智能体是否记住了该事实。

In [ ]:
def run_recall_eval(window_size: int) -> dict:
    """在给定窗口大小下运行回忆评估，返回得分。"""
    mem = SlidingWindowMemory(
        model=llm,
        window_size=window_size,
        system_prompt="You are a helpful assistant. Answer concisely.",
    )

    # 阶段 1：埋入事实（5 轮 = 10 条消息）
    for fact, _ in FACTS:
        mem.chat(fact)

    # 阶段 2：用填充轮次把旧事实挤出窗口（5 轮 = 10 条消息）
    for filler in FILLER_MESSAGES:
        mem.chat(filler)

    # 阶段 3：提出回忆问题并检查答案
    results = {}
    for question, keyword in RECALL_QUESTIONS:
        answer = mem.chat(question)
        recalled = keyword.lower() in answer.lower()
        results[keyword] = {
            "question": question,
            "answer": answer[:100],
            "recalled": recalled,
        }

    score = sum(1 for r in results.values() if r["recalled"])
    return {
        "window_size": window_size,
        "score": score,
        "total": len(RECALL_QUESTIONS),
        "pct": score / len(RECALL_QUESTIONS) * 100,
        "details": results,
        "total_input_tokens": sum(t["input_tokens"] for t in mem.turn_token_usage),
    }




在 4、8、12、20 条消息四种窗口大小下运行评估。较小的窗口会忘记对话早期埋下的事实；较大的窗口能回忆更多，token 成本也更高。

In [ ]:
# 测试多个窗口大小
WINDOW_SIZES = [4, 8, 12, 20]
eval_results = []

for k in WINDOW_SIZES:
    print(f"Running eval with window_size={k}...", end=" ")
    result = run_recall_eval(k)
    eval_results.append(result)
    print(f"Recall: {result['score']}/{result['total']} ({result['pct']:.0f}%)")

print("\nDone!")

绘制结果。左图是每个窗口大小下的回忆准确率，右图是总 token 成本。你会看到核心权衡：窗口越大，记得越多，花的 token 也越多。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

window_sizes = [r["window_size"] for r in eval_results]
scores = [r["pct"] for r in eval_results]
tokens = [r["total_input_tokens"] for r in eval_results]

# 左图：回忆准确率随窗口大小的变化
colors = ["#ef4444" if s < 40 else "#f59e0b" if s < 80 else "#22c55e" for s in scores]
bars = ax1.bar(range(len(window_sizes)), scores, color=colors, alpha=0.85, width=0.6)
ax1.set_xticks(range(len(window_sizes)))
ax1.set_xticklabels([f"k={k}" for k in window_sizes])
ax1.set_ylabel("Recall Accuracy (%)")
ax1.set_title("Recall Accuracy vs. Window Size")
ax1.set_ylim(0, 110)
ax1.axhline(y=100, color="gray", linestyle="--", alpha=0.3)
for bar, score in zip(bars, scores):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f"{score:.0f}%", ha="center", va="bottom", fontweight="bold")

# 右图：总 token 成本随窗口大小的变化
ax2.plot(window_sizes, tokens, "o-", color="#6366f1", linewidth=2, markersize=8)
ax2.fill_between(window_sizes, tokens, alpha=0.1, color="#6366f1")
ax2.set_xlabel("Window Size (k messages)")
ax2.set_ylabel("Total Input Tokens")
ax2.set_title("Total Token Cost vs. Window Size")
for x, y in zip(window_sizes, tokens):
    ax2.annotate(f"{y:,}", (x, y), textcoords="offset points",
                 xytext=(0, 10), ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("sliding_window_eval.png", dpi=150, bbox_inches="tight")
plt.show()

print("The tradeoff is clear: larger windows recall more but cost more tokens.")

构建一个回忆矩阵，展示每个窗口大小下哪些事实被保住。这张表清楚地说明：越早埋入的事实，在窗口缩小时越先被遗忘。

In [ ]:
import pandas as pd

# 构建汇总表
rows = []
for r in eval_results:
    for keyword, detail in r["details"].items():
        rows.append({
            "Window Size": r["window_size"],
            "Fact": keyword,
            "Recalled?": "✓" if detail["recalled"] else "✗",
            "Answer (preview)": detail["answer"][:60],
        })

df = pd.DataFrame(rows)
# 透视：事实作列，窗口大小作行
pivot = df.pivot(index="Window Size", columns="Fact", values="Recalled?")
print("Recall matrix (✓ = remembered, ✗ = forgotten):\n")
print(pivot.to_string())

## Token 成本：滑动窗口 vs. 完整缓冲

滑动窗口的全部意义在于**有界且可预测的成本**。我们把两条 token 增长曲线放在一起比较。

In [ ]:
# 模拟完整缓冲与滑动窗口的 token 增长
# 简单模型：每条消息约 50 token，系统提示词约 30 token
MSG_TOKENS = 50
SYS_TOKENS = 30
NUM_TURNS = 25
WINDOW_K = 10  # 5 个轮次

buffer_input = []
window_input = []

for turn in range(1, NUM_TURNS + 1):
    n_msgs = turn * 2  # 每轮一条用户消息 + 一条助手消息
    buffer_input.append(SYS_TOKENS + n_msgs * MSG_TOKENS)
    window_msgs = min(n_msgs, WINDOW_K)
    window_input.append(SYS_TOKENS + window_msgs * MSG_TOKENS)

turns = list(range(1, NUM_TURNS + 1))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(turns, buffer_input, "o-", color="#ef4444", label="Full Buffer", linewidth=2)
ax.plot(turns, window_input, "s-", color="#22c55e", label=f"Sliding Window (k={WINDOW_K})", linewidth=2)
ax.fill_between(turns, window_input, buffer_input, alpha=0.08, color="#ef4444")
ax.set_xlabel("Conversation Turn")
ax.set_ylabel("Input Tokens per API Call")
ax.set_title("Input Tokens per Turn: Full Buffer vs. Sliding Window")
ax.legend()
ax.grid(True, alpha=0.2)

# 在最后一轮标注节省量
savings = buffer_input[-1] - window_input[-1]
ax.annotate(
    f"Savings: {savings:,} tokens/turn",
    xy=(NUM_TURNS, (buffer_input[-1] + window_input[-1]) / 2),
    fontsize=11, fontweight="bold", color="#6366f1",
    ha="right",
)

plt.tight_layout()
plt.savefig("buffer_vs_window.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"At turn {NUM_TURNS}:")
print(f"  Full buffer:    {buffer_input[-1]:,} input tokens")
print(f"  Sliding window: {window_input[-1]:,} input tokens")
print(f"  Savings:        {savings:,} tokens ({savings/buffer_input[-1]*100:.0f}%)")

## 讨论与权衡

### 优点
- **成本可预测**：无论对话多长，每轮 token 用量都由窗口大小封顶。
- **构建快**：Python 的 `deque(maxlen=k)` 一行代码就实现了全部机制。
- **对大多数聊天足够用**：用户很少提到 20 多轮之前的内容。
- **低延迟**：输入大小有界，响应时间也就有界。

### 缺点
- **硬性的信息截断**：消息一旦滑出窗口就没了。没有痕迹、没有摘要，什么都没有。
- **没有优雅降级**：与摘要记忆不同，智能体甚至不知道自己忘了东西。
- **孤立消息**：按消息计数的窗口可能把一组「用户-助手」对话对从中间切开，留下只剩回答、没有提问的情况。（我们的实现按消息计数并避免了这个问题，按轮次计数是另一种选择。）
- **选择 *k* 很讲究**：太小会丢掉重要上下文，太大就失去了这个方法的意义。

### 如何选择窗口大小

| 窗口大小 | 适用场景 | 注意事项 |
|-------------|----------|-----------|
| k = 4-6 条消息 | 快速问答、无状态任务 | 2-3 轮后忘记上下文 |
| k = 10-20 条消息 | 大多数聊天机器人、客服 | 成本上升；仍可能忘记早期事实 |
| k = 40+ 条消息 | 复杂的多步任务 | 成本接近缓冲记忆；考虑混合方案 |

### 经验法则
从 `k = 10`（5 个完整轮次）开始，再根据回忆需求和预算调整。

### 什么时候该换别的方法
- 需要记住更早的事实：用**摘要记忆**或**实体记忆**。
- 需要平衡新近度和成本，又不能接受硬截断：用**摘要 + 缓冲混合**。
- 需要严格控制在精确的 token 预算内：用 **token 缓冲记忆**。

## 延伸阅读

- [Anthropic Messages API：多轮对话](https://docs.anthropic.com/en/docs/build-with-claude/conversational-ai?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)
- [LangChain ConversationBufferWindowMemory](https://python.langchain.com/docs/modules/memory/types/buffer_window?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)
- [LlamaIndex ChatMemoryBuffer](https://docs.llamaindex.ai/en/stable/api_reference/memory/chat_memory_buffer/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)
- [Lilian Weng，《LLM Powered Autonomous Agents》（记忆部分）](https://lilianweng.github.io/posts/2023-06-23-agent/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)
- Python `collections.deque`：[官方文档](https://docs.python.org/3/library/collections.html#collections.deque)

---

*← 上一篇：01 对话缓冲记忆 · 下一篇：[03 摘要记忆](../03_summary_memory/) →*

In [ ]:
# 清理演示生成的临时文件
import os
for f in ["sliding_window_eval.png", "buffer_vs_window.png"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")

## 🧪 动手试试

三个小挑战，帮你加深理解。每个大约需要 10-30 分钟。

### 挑战 1：淘汰回调
给 `SlidingWindowMemory` 添加一个 `on_evict` 回调：每淘汰一条消息，就用该消息调用一次回调。把被淘汰的消息记录到一个列表里，最后打印出来。这样你就能看清智能体究竟忘了什么。

### 挑战 2：窗口大小与回忆热力图
用窗口大小 [2, 4, 6, 8, 10, 16, 20] 运行 `run_recall_eval()`，绘制一张热力图，展示不同窗口大小、不同问题类别下的回忆得分。找出对身份类问题回忆准确率不低于 80% 的最小窗口。

### 挑战 3：带摘要兜底的滑动窗口
每当有消息从 deque 中淘汰，就把它交给 LLM 更新一份滚动摘要（类似 03 摘要记忆）。每一轮都把这份摘要放在上下文最前面。测量这种混合方案在回忆基准测试上能否胜过纯滑动窗口。


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--02-sliding-window-memory--sliding-window-memory)
